# 01 · Data Understanding, Cleaning and Preparation

**Purpose:** Validate the source extract, document its long-format structure, prepare the obesity-only analytical scope, and export reusable processed datasets.

This notebook preserves the substantive code and analytical sequence from the original completed project. Changes are limited to portable file paths, organization, and explanatory markdown.


## Reproducible setup

The project-relative paths below replace the original Google Colab mount so the notebook runs consistently in VS Code, Jupyter, or GitHub Codespaces.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA = PROJECT_ROOT / "data" / "raw" / "cdc_brfss_nutrition_activity_obesity_2011_2024.csv"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed" / "obesity_analysis_ready.csv"


## 1. Data Dictionary

This section provides an overview of the dataset columns, their descriptions, and data types (based on initial inspection).

| Column | Description | Data Type | Non-Null Count |
|--------|-------------|-----------|----------------|
| YearStart | Starting year of the data record | int64 | 110880 |
| YearEnd | Ending year of the data record | int64 | 110880 |
| LocationAbbr | Abbreviation of the location (e.g., state) | object | 110880 |
| LocationDesc | Full description of the location | object | 110880 |
| Datasource | Source of the data (e.g., BRFSS) | object | 110880 |
| Class | Category of the health measure (e.g., Obesity / Weight Status) | object | 110880 |
| Topic | Specific topic within the class (e.g., Obesity) | object | 110880 |
| Question | The survey question | object | 110880 |
| Data_Value_Unit | Unit of the data value (e.g., %) | object | 4620 |
| Data_Value_Type | Type of value (e.g., Percentage) | object | 110880 |
| Data_Value | Main value (e.g., obesity prevalence %) | float64 | 97666 |
| Data_Value_Alt | Alternative representation of Data_Value | float64 | 97666 |
| Data_Value_Footnote_Symbol | Symbol for footnotes | object | 13214 |
| Data_Value_Footnote | Footnote text | object | 13214 |
| Low_Confidence_Limit | Lower confidence interval | float64 | 97666 |
| High_Confidence_Limit | Upper confidence interval | float64 | 97666 |
| Sample_Size | Sample size for the estimate | object | 97666 |
| Total | Total category (for unstratified) | object | 3960 |
| Age(years) | Age stratification | object | 23760 |
| Education | Education stratification | object | 15840 |
| Sex | Sex stratification | object | 7920 |
| Income | Income stratification | object | 27720 |
| Race/Ethnicity | Race/Ethnicity stratification | object | 31680 |
| GeoLocation | Geographic coordinates | object | 108864 |
| ClassID | ID for Class | object | 110880 |
| TopicID | ID for Topic | object | 110880 |
| QuestionID | ID for Question | object | 110880 |
| DataValueTypeID | ID for Data_Value_Type | object | 110880 |
| LocationID | ID for Location | int64 | 110880 |
| StratificationCategory1 | Primary stratification category (e.g., Age, Income) | object | 110880 |
| Stratification1 | Specific stratification value | object | 110880 |
| StratificationCategoryId1 | ID for StratificationCategory1 | object | 110880 |
| StratificationID1 | ID for Stratification1 | object | 110880 |

## 2. Data Understanding & Preparation

### 2.1 Data Understanding

#### Import Libraries

**What this code does:** Tukey HSD identifies which group pairs differ after the overall ANOVA while controlling the family-wise error rate.


In [ ]:
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

print("Core analysis libraries loaded.")


Load Dataset

**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
# Load the original CDC BRFSS extract using a portable project-relative path.
data = pd.read_csv(RAW_DATA, low_memory=False)
data.columns = data.columns.str.strip().str.replace(r"\s+", "_", regex=True)
print(f"Loaded {len(data):,} raw rows and {data.shape[1]} columns.")


#### Dataset Overview


**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
# Inspect dataset structure and data types

data.info()

**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
#Show shape

print(f"Shape of the dataset: {data.shape}")

**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
#show first 5 rows

data.head(5)

#### Missing values Overview

**Why this step matters:** Missingness can distort comparisons. This check separates true analytical gaps from fields that are blank by design in the BRFSS long format.


In [ ]:
# Assess missing values to distinguish structural vs analytical gaps

missing_info = pd.DataFrame({
    "Missing Count": data.isnull().sum(),
    "Missing Percentage": (data.isnull().sum() / len(data)) * 100
}).query("`Missing Count` > 0").sort_values("Missing Percentage", ascending=False)

missing_info

**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
data['Income']

Check for Duplicate Rows

**Why this step matters:** Duplicate records can over-weight specific location and subgroup estimates, so they are checked before analysis.


In [ ]:
# Check for duplicate rows

print(f"Number of duplicated rows: {data.duplicated().sum()}")

#### Column roles and BRFSS long-format structure

Each row represents one health measure for one location and year, reported for a single population stratification.

- Outcome: Data_Value (obesity prevalence %)
- Reliability: Sample_Size, Low_Confidence_Limit, High_Confidence_Limit
- Identifiers: Class, Topic, Question, and IDs
- Time & Geography: YearStart, YearEnd, LocationAbbr, LocationDesc
- Stratification: StratificationCategory1, Stratification1

High missingness in stratification columns is expected due to long-format reporting (one stratification per row).


Cardinality / Uniqueness Check

**What this code does:** Reviews field cardinality to identify identifiers, categories, and columns that may need special treatment.


In [ ]:
# Cardinality (unique values) check
unique_values = data.nunique()
unique_values

Stratification Structure Analysis

**Why this step matters:** Missingness can distort comparisons. This check separates true analytical gaps from fields that are blank by design in the BRFSS long format.


In [ ]:
strat_map = {
    "Income": "Income",
    "Education": "Education",
    "Sex": "Sex",
    "Race/Ethnicity": "Race/Ethnicity",
    "Age (years)": "Age(years)"
}

strat_counts = data["StratificationCategory1"].value_counts(dropna=False)

# Missing rate in expected column for each stratification
rows = []
for strat_cat, expected_col in strat_map.items():
    sub = data[data["StratificationCategory1"] == strat_cat]
    if len(sub) > 0:
        missing_rate = (sub[expected_col].isna().mean() * 100).round(2)
        rows.append({"StratificationCategory1": strat_cat, "expected_column": expected_col,
                     "rows_in_category": len(sub), "missing_rate_in_expected_column_%": missing_rate})

structural_missingness_check = pd.DataFrame(rows).sort_values("missing_rate_in_expected_column_%", ascending=False)
structural_missingness_check

Validity Checks for Numeric Fields

**Why this step matters:** Confidence-interval width indicates estimate precision and helps prevent over-interpreting uncertain subgroup estimates.


In [ ]:
# Percentage range check (0-100)
def pct_out_of_range(df, col):
    if col not in df.columns: return pd.DataFrame()
    bad = df[df[col].notna() & ((df[col] < 0) | (df[col] > 100))].copy()
    keep_cols = ["YearStart", "LocationAbbr", "QuestionID", col]
    keep_cols = [c for c in keep_cols if c in bad.columns]
    return bad[keep_cols]

print("Out-of-range Data_Value rows:", len(pct_out_of_range(data, "Data_Value")))               # 0
print("Out-of-range Low_Confidence_Limit rows:", len(pct_out_of_range(data, "Low_Confidence_Limit")))  # 0
print("Out-of-range High_Confidence_Limit rows:", len(pct_out_of_range(data, "High_Confidence_Limit"))) # 0

# Confidence interval logic check
ci_issues = data[
    data["Data_Value"].notna() &
    data["Low_Confidence_Limit"].notna() &
    data["High_Confidence_Limit"].notna() &
    ((data["Low_Confidence_Limit"] > data["High_Confidence_Limit"]) |
     (data["Data_Value"] < data["Low_Confidence_Limit"]) |
     (data["Data_Value"] > data["High_Confidence_Limit"]))
]
print("CI logic issue rows:", len(ci_issues))  # 0

Uniqueness Check Using Candidate Key


**Why this step matters:** Duplicate records can over-weight specific location and subgroup estimates, so they are checked before analysis.


In [ ]:
candidate_key = ["YearStart", "LocationAbbr", "QuestionID", "StratificationCategory1", "Stratification1"]
dup_by_key = data.duplicated(subset=candidate_key).sum()
dup_by_key

### 2.2 Data Preparation
#### Type conversion and standardization


**Why this step matters:** Confidence-interval width indicates estimate precision and helps prevent over-interpreting uncertain subgroup estimates.


In [ ]:
# remove spaces of column names
data.columns = data.columns.str.strip()

# Standardize data types for analysis

text_cols = [
    "LocationAbbr", "LocationDesc", "Datasource", "Class", "Topic", "Question",
    "Data_Value_Type", "Data_Value_Unit",
    "StratificationCategory1", "Stratification1",
    "Total", "Age(years)", "Education", "Sex", "Income", "Race/Ethnicity"
]
for col in text_cols:
    if col in data.columns:
        data[col] = data[col].astype("string").str.strip()

numeric_cols = [
    'YearStart', 'YearEnd',
    'Data_Value', 'Data_Value_Alt',
    'Low_Confidence_Limit', 'High_Confidence_Limit',
    'Sample_Size'
]

for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')


**Why this step matters:** Missingness can distort comparisons. This check separates true analytical gaps from fields that are blank by design in the BRFSS long format.


In [ ]:
# calculate missing rate in the expected column
rows = []

for strat_cat, expected_col in strat_map.items():
    # skip if columns are missing
    if "StratificationCategory1" not in data.columns or expected_col not in data.columns:
        continue

    # subset rows for this stratification category
    sub = data[data["StratificationCategory1"] == strat_cat]
    if len(sub) == 0:
        continue

    missing_rate = (sub[expected_col].isna().mean() * 100).round(2)

    rows.append({
        "StratificationCategory1": strat_cat,
        "expected_column": expected_col,
        "rows_in_category": len(sub),
        "missing_rate_in_expected_column_%": missing_rate
    })
missing_rate

Summary Statistics for Key Numeric Variables

**Why this step matters:** Confidence-interval width indicates estimate precision and helps prevent over-interpreting uncertain subgroup estimates.


In [ ]:
# Summary statistics for key numeric variables
data[[
    "Data_Value",
    "Sample_Size",
    "Low_Confidence_Limit",
    "High_Confidence_Limit"
]].describe()


#### Analytical Scoping & Dataset Filtering


**What this step does:** Executes the next stage of the original analysis and preserves its result for the interpretation that follows.


In [ ]:
# Define the target obesity question
target_question = "Percent of adults aged 18 years and older who have obesity"

# Filter dataset to the selected question
df_obesity = data[
    (data["Question"] == target_question) &
    (data["Data_Value"].notna())
].copy()

print("Filtered dataset shape:", df_obesity.shape)

# Filter to income-based stratification
df_income = df_obesity[
    df_obesity["StratificationCategory1"] == "Income"
].copy()

print("Income-stratified dataset shape:", df_income.shape)

####Export Clean, Analysis-Ready Files

**What this code does:** Reshapes the long survey data into a comparison-friendly table without changing the underlying estimates.


In [ ]:
# Export the clean long-format core file used by the downstream notebooks.
output_path = PROJECT_ROOT / "data" / "processed" / "obesity_analysis_ready.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df_obesity.to_csv(output_path, index=False)

# Preserve the original income-focused wide export for convenient comparison.
if "df_income" in globals() and not df_income.empty:
    clean_income_wide = df_income.pivot_table(
        index=["YearStart", "LocationDesc"],
        columns="Stratification1",
        values="Data_Value",
        aggfunc="mean",
    ).reset_index()
    clean_income_wide.to_csv(PROJECT_ROOT / "data" / "processed" / "income_comparison_wide.csv", index=False)

print(f"Saved analysis-ready data to {output_path.relative_to(PROJECT_ROOT)}")
